S2/S3: https://browser.dataspace.copernicus.eu/
Download the specific images post login.

In [1]:
# Import necessary libraries
import os
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.geometry import Point
import sys
import zipfile
import numpy as np

# Debug: Confirm that imports are successful
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module
print(f"Python version: {sys.version}")
print(f"os version: Part of Python standard library, version {sys.version}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"rasterio version: {rasterio.__version__}")
print(f"zipfile version: Part of Python standard library, version {sys.version}")

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
os version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
numpy version: 1.26.4
pandas version: 2.2.3
geopandas version: 0.14.4
rasterio version: 1.4.3
zipfile version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]


In [2]:
# Set the base directory for datasets in Kaggle
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets
sub_dir = r"/kaggle/working/"  # Submission directory for output files

# Debug: Print the directory paths to confirm they are set correctly
print(f"Debug: Base directory: {base_dir}")
print(f"Debug: Submission directory: {sub_dir}")

# Debug: List the files in the base directory to verify the presence of input files
print(f"Debug: Listing files in {base_dir}:")
try:
    files_in_dir = os.listdir(base_dir)
    print(files_in_dir)
except Exception as e:
    print(f"Error: Could not list files in {base_dir}. Error: {str(e)}")
    raise Exception(f"Failed to access directory {base_dir}")

# File paths for input datasets
TRAIN_FILE = os.path.join(base_dir, "Training_data.csv")
VALIDATION_FILE = os.path.join(base_dir, "Validation_data.csv")

# Paths to TIFF files
S2_FILE = os.path.join(base_dir, "S2_DATA.tiff")
LST_FILE = os.path.join(base_dir, "Landsat_LST.tiff")
LST_NDVI_FILE = os.path.join(base_dir, "Landsat_NDVI.tiff")
LANDSAT9_REFL = os.path.join(base_dir, "LSAT_8_221022", "LC09_L1TP_140046_20240102_20240102_02_T1_refl.tif")
LANDSAT8_ST_B10 = os.path.join(base_dir, "LSAT_8_221022", "LC08_L2SP_013032_20221009_20221013_02_T1_ST_B10.TIF")

# Sentinel-2 and Sentinel-3 directories
SENTINEL2_DIR = os.path.join(base_dir, "Sentinel2")
SENTINEL3_DIR = os.path.join(base_dir, "Sentinel_3")

# Final output CSVs for training and validation with satellite data
TRAIN_OUTPUT = os.path.join(sub_dir, "training_data_with_satellite_data.csv")
VALIDATION_OUTPUT = os.path.join(sub_dir, "validation_data_with_satellite_data.csv")

# Debug: Check if input files exist
print(f"Debug: Training dataset file exists: {os.path.exists(TRAIN_FILE)}")
print(f"Debug: Validation dataset file exists: {os.path.exists(VALIDATION_FILE)}")
print(f"Debug: S2_DATA.tiff exists: {os.path.exists(S2_FILE)}")
print(f"Debug: Landsat_LST.tiff exists: {os.path.exists(LST_FILE)}")
print(f"Debug: Landsat_NDVI.tiff exists: {os.path.exists(LST_NDVI_FILE)}")
print(f"Debug: Landsat9 reflectance file exists: {os.path.exists(LANDSAT9_REFL)}")
print(f"Debug: Landsat8 ST B10 file exists: {os.path.exists(LANDSAT8_ST_B10)}")
print(f"Debug: Sentinel-2 directory exists: {os.path.exists(SENTINEL2_DIR)}")
print(f"Debug: Sentinel-3 directory exists: {os.path.exists(SENTINEL3_DIR)}")

Debug: Base directory: /kaggle/input/eyds-base-dataset
Debug: Submission directory: /kaggle/working/
Debug: Listing files in /kaggle/input/eyds-base-dataset:
['census_block_loc.csv', 'Hyperlocal_Temperature_Monitoring_20250311.csv', 'Airquality_Unique_geocode_with_LatLong.xlsx', 'nyclion_25a', 'StreetAssessmentRating', 'Sentinel2', 'USA_wind-speed_10m.tif', 'USA_power-density_10m.tif', 'Validation_data.csv', 'LSAT_8_221022', 'Training_data.csv', 'NYC_Cooling_Tower_Registrations_20250224.csv', 'AQ', 'S2_DATA.tiff', 'Automated_Traffic_Volume_Counts_20250319.csv', 'Air_Quality_20250221.csv', 'USA_air-density_10m.tif', 'nclimgrid-monthly-202107.tif', 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250224.csv', 'Building_Footprint.kml', 'Sentinel_3', 'Landsat_NDVI.tiff', '2015_Street_Tree_Census_-_Tree_Data_20250221.csv', 'NY_Mesonet_Weather.xlsx', 'Landsat_LST.tiff', 'Building Footprints_20250222.geojson', 'nyc_census_tracts.csv']
Debug: Training dat

In [3]:
# Define Sentinel-2 filenames (as provided)
sentinel2_filenames = [
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color_(urban).tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Highlight_Optimized_Natural_Color_.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Moisture_index.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDSI.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDVI.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDWI.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Scene_classification_map_.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_SWIR.tiff",
    "2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_True_color.tiff"
]

# Define Sentinel-3 filenames (as provided)
sentinel3_filenames = [
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_F1_Brightness_Temperature.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_F2_Brightness_Temperature.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_False_Color.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S1_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S2_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S2_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S3_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S4_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S4_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S5_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S5_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S6_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S6_Reflectance_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S7_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S7_Brightness_Temperature_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S8_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S8_Brightness_Temperature_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S9_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S9_Brightness_Temperature_.tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S3_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_S1_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_F2_(Raw).tiff",
    "2021-07-24-00_00_2021-07-24-23_59_Sentinel-3_SLSTR_F1_(Raw).tiff"
]

# Create full paths for Sentinel-2 and Sentinel-3 files
sentinel2_files = [os.path.join(SENTINEL2_DIR, fname) for fname in sentinel2_filenames]
sentinel3_files = [os.path.join(SENTINEL3_DIR, fname) for fname in sentinel3_filenames]

# Debug: Check if Sentinel-2 and Sentinel-3 files exist
print("Debug: Checking Sentinel-2 files:")
for file in sentinel2_files:
    print(f"File exists: {file} - {os.path.exists(file)}")

print("Debug: Checking Sentinel-3 files:")
for file in sentinel3_files:
    print(f"File exists: {file} - {os.path.exists(file)}")

Debug: Checking Sentinel-2 files:
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color_(urban).tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Highlight_Optimized_Natural_Color_.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Moisture_index.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDSI.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDVI.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDWI.tiff - True
File exists: /kaggle/input/eyds-base-dataset/Sentinel2/20

In [4]:
#############################################
# Part A: Load Training and Validation Data
#############################################

# 1. Load the training and validation datasets and convert them to GeoDataFrames
print("Loading training data...")
try:
    train_df = pd.read_csv(TRAIN_FILE)
except FileNotFoundError:
    print(f"Error: Training data file not found at {TRAIN_FILE}")
    raise Exception("Failed to load training dataset")

print("Training data loaded with", train_df.shape[0], "rows.")
gdf_train = gpd.GeoDataFrame(
    train_df.copy(),
    geometry=gpd.points_from_xy(train_df.Longitude, train_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Training GeoDataFrame shape: {gdf_train.shape}")
print(f"Debug: Training GeoDataFrame columns: {gdf_train.columns.tolist()}")

print("Loading validation data...")
try:
    val_df = pd.read_csv(VALIDATION_FILE)
except FileNotFoundError:
    print(f"Error: Validation data file not found at {VALIDATION_FILE}")
    raise Exception("Failed to load validation dataset")

print("Validation data loaded with", val_df.shape[0], "rows.")
gdf_val = gpd.GeoDataFrame(
    val_df.copy(),
    geometry=gpd.points_from_xy(val_df.Longitude, val_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Validation GeoDataFrame shape: {gdf_val.shape}")
print(f"Debug: Validation GeoDataFrame columns: {gdf_val.columns.tolist()}")

Loading training data...
Training data loaded with 11229 rows.
Debug: Training GeoDataFrame shape: (11229, 5)
Debug: Training GeoDataFrame columns: ['Longitude', 'Latitude', 'datetime', 'UHI Index', 'geometry']
Loading validation data...
Validation data loaded with 1040 rows.
Debug: Validation GeoDataFrame shape: (1040, 4)
Debug: Validation GeoDataFrame columns: ['Longitude', 'Latitude', 'UHI Index', 'geometry']


In [5]:
#############################################
# Part B: Define Sampling Function
#############################################

# 2. Function to sample raster values at point locations from a specified band
def sample_raster_values(raster_path, gdf, col_name, band_index=1):
    try:
        with rasterio.open(raster_path) as src:
            # Reproject GeoDataFrame if CRS doesn't match
            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)
            # Prepare list of (x, y) coordinates
            coords = [(point.x, point.y) for point in gdf.geometry]
            # Sample the specified band (band_index - 1 because of 0-indexing)
            values = [val[band_index - 1] for val in src.sample(coords)]
            gdf[col_name] = values
    except Exception as e:
        print(f"Error processing {raster_path}: {e}")
        gdf[col_name] = [None] * len(gdf)  # Fill with None if there's an error
    return gdf

In [6]:
#############################################
# Part C: Extract Satellite Data
#############################################

# 3. Extract values from S2_DATA.tiff, Landsat_LST.tiff, and Landsat_NDVI.tiff
print("Extracting s2_value, lst_value, and lst_value_ndvi...")
gdf_train = sample_raster_values(S2_FILE, gdf_train, "s2_value", band_index=1)
gdf_val = sample_raster_values(S2_FILE, gdf_val, "s2_value", band_index=1)

gdf_train = sample_raster_values(LST_FILE, gdf_train, "lst_value", band_index=1)
gdf_val = sample_raster_values(LST_FILE, gdf_val, "lst_value", band_index=1)

gdf_train = sample_raster_values(LST_NDVI_FILE, gdf_train, "lst_value_ndvi", band_index=1)
gdf_val = sample_raster_values(LST_NDVI_FILE, gdf_val, "lst_value_ndvi", band_index=1)

# 4. Extract Landsat data (L8_ST_B10_raw, L8_ST_B10_C, and L9_refl_phys)
print("Extracting Landsat data...")
# Landsat 9 Reflectance (band 3 for red band)
gdf_train = sample_raster_values(LANDSAT9_REFL, gdf_train, "L9_refl_raw", band_index=3)
gdf_val = sample_raster_values(LANDSAT9_REFL, gdf_val, "L9_refl_raw", band_index=3)

# Convert to physical reflectance
gdf_train["L9_refl_phys"] = gdf_train["L9_refl_raw"] * 0.0001
gdf_val["L9_refl_phys"] = gdf_val["L9_refl_raw"] * 0.0001

# Landsat 8 Thermal Band (B10)
gdf_train = sample_raster_values(LANDSAT8_ST_B10, gdf_train, "L8_ST_B10_raw", band_index=1)
gdf_val = sample_raster_values(LANDSAT8_ST_B10, gdf_val, "L8_ST_B10_raw", band_index=1)

# Convert to Celsius
gdf_train["L8_ST_B10_C"] = gdf_train["L8_ST_B10_raw"].apply(lambda x: x * 0.00341802 - 273.15 if x is not None else None)
gdf_val["L8_ST_B10_C"] = gdf_val["L8_ST_B10_raw"].apply(lambda x: x * 0.00341802 - 273.15 if x is not None else None)

# 5. Extract Sentinel-2 data
print("Extracting Sentinel-2 data...")
sentinel2_col_names = [
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_False_color",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_False_color_urban",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Highlight_Optimized_Natural_Color_",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Moisture_index",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDSI",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDVI",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDWI",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Scene_classification_map_",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_SWIR",
    "2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_True_color"
]

for file_path, col_name in zip(sentinel2_files, sentinel2_col_names):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        gdf_train[col_name] = [None] * len(gdf_train)
        gdf_val[col_name] = [None] * len(gdf_val)
        continue
    print(f"Processing Sentinel-2 file: {file_path}")
    gdf_train = sample_raster_values(file_path, gdf_train, col_name, band_index=1)
    gdf_val = sample_raster_values(file_path, gdf_val, col_name, band_index=1)

# 6. Extract Sentinel-3 data
print("Extracting Sentinel-3 data...")
sentinel3_col_names = [
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_F1_Brightness_Temperature",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_F2_Brightness_Temperature",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_False_Color",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S1_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S2_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S2_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S3_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S4_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S4_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S5_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S5_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S6_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S6_Reflectance_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S7_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S7_Brightness_Temperature_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S8_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S8_Brightness_Temperature_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S9_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S9_Brightness_Temperature_",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S3_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S1_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_F2_Raw",
    "2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_F1_Raw"
]

for file_path, col_name in zip(sentinel3_files, sentinel3_col_names):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        gdf_train[col_name] = [None] * len(gdf_train)
        gdf_val[col_name] = [None] * len(gdf_val)
        continue
    print(f"Processing Sentinel-3 file: {file_path}")
    gdf_train = sample_raster_values(file_path, gdf_train, col_name, band_index=1)
    gdf_val = sample_raster_values(file_path, gdf_val, col_name, band_index=1)

Extracting s2_value, lst_value, and lst_value_ndvi...
Extracting Landsat data...
Extracting Sentinel-2 data...
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color.tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_False_color_(urban).tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Highlight_Optimized_Natural_Color_.tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_Moisture_index.tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDSI.tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-dataset/Sentinel2/2021-08-25-00_00_2021-08-25-23_59_Sentinel-2_L2A_NDVI.tiff
Processing Sentinel-2 file: /kaggle/input/eyds-base-d

In [7]:
#############################################
# Part D: Save the Enhanced DataFrames
#############################################

# 7. Convert GeoDataFrames back to regular DataFrames and save
train_enhanced = pd.DataFrame(gdf_train.drop(columns="geometry"))
val_enhanced = pd.DataFrame(gdf_val.drop(columns="geometry"))

# Debug: Check the final DataFrames
print("Debug: Final training data shape:", train_enhanced.shape)
print("Debug: Final training data columns:", train_enhanced.columns.tolist())
print("Debug: Sample final training data (first 5 rows):")
print(train_enhanced[['Longitude', 'Latitude', 's2_value', 'lst_value', 'lst_value_ndvi', 'L8_ST_B10_raw', 'L8_ST_B10_C']].head())

print("Debug: Final validation data shape:", val_enhanced.shape)
print("Debug: Final validation data columns:", val_enhanced.columns.tolist())
print("Debug: Sample final validation data (first 5 rows):")
print(val_enhanced[['Longitude', 'Latitude', 's2_value', 'lst_value', 'lst_value_ndvi', 'L8_ST_B10_raw', 'L8_ST_B10_C']].head())

# 8. Save the enhanced DataFrames to CSV
train_enhanced.to_csv(TRAIN_OUTPUT, index=False)
val_enhanced.to_csv(VALIDATION_OUTPUT, index=False)

# Debug: Print the final confirmation messages with file paths
print(f"Debug: Training data with satellite features saved to: {TRAIN_OUTPUT}")
print(f"Debug: Validation data with satellite features saved to: {VALIDATION_OUTPUT}")
print("Done! Satellite data columns added to both training and validation datasets.")
print(f"Train output: {TRAIN_OUTPUT}")
print(f"Validation output: {VALIDATION_OUTPUT}")

Debug: Final training data shape: (11229, 44)
Debug: Final training data columns: ['Longitude', 'Latitude', 'datetime', 'UHI Index', 's2_value', 'lst_value', 'lst_value_ndvi', 'L9_refl_raw', 'L9_refl_phys', 'L8_ST_B10_raw', 'L8_ST_B10_C', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_False_color', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_False_color_urban', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Highlight_Optimized_Natural_Color_', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Moisture_index', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDSI', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDVI', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_NDWI', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_Scene_classification_map_', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_SWIR', '2021_08_25_00_00_2021_08_25_23_59_Sentinel_2_L2A_True_color', '2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_F1_Brightness_Temperature', '2021_07_24_00_00_2021_0